In [116]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

In [117]:
df = pd.read_csv('diabetes.csv')

In [118]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [119]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [120]:
df.shape

(768, 9)

In [121]:
df.corr()['Outcome']

Pregnancies                 0.221898
Glucose                     0.466581
BloodPressure               0.065068
SkinThickness               0.074752
Insulin                     0.130548
BMI                         0.292695
DiabetesPedigreeFunction    0.173844
Age                         0.238356
Outcome                     1.000000
Name: Outcome, dtype: float64

In [122]:
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

In [123]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [124]:
X = scaler.fit_transform(X)

In [125]:
X.shape

(768, 8)

In [126]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size= 0.2,
    random_state= 1
)

In [127]:
import tensorflow as tf
from tensorflow import keras 
from keras import Sequential 
from keras.layers import Dense

In [128]:
model = Sequential()
model.add(Dense (32, activation= 'relu', input_dim = 8))
model.add(Dense(1, activation= 'sigmoid'))

In [129]:
model.compile(optimizer= 'Adam', loss = 'binary_crossentropy', metrics= ['accuracy'])

In [130]:
model.fit(X_train, y_train, batch_size= 32, epochs= 100, validation_data=(X_test, y_test))

Epoch 1/100


20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6482 - loss: 0.6438 - val_accuracy: 0.6558 - val_loss: 0.6186
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6889 - loss: 0.6053 - val_accuracy: 0.7338 - val_loss: 0.5864
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7101 - loss: 0.5760 - val_accuracy: 0.7727 - val_loss: 0.5612
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7280 - loss: 0.5532 - val_accuracy: 0.7792 - val_loss: 0.5423
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7313 - loss: 0.5349 - val_accuracy: 0.7792 - val_loss: 0.5274
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7410 - loss: 0.5213 - val_accuracy: 0.7857 - val_loss: 0.5142
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7476 - loss: 0.5098 - val_accuracy: 0.7922 - val_loss: 0.5074
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7557 - loss: 0.5002 - val_accuracy: 0.7987 - val_loss: 0.

In [131]:
import kerastuner as kt

In [132]:
def build_model(hp):

    model = Sequential()

    model.add(Dense(32, activation= 'relu', input_dim = 8))
    model.add(Dense(1, activation='sigmoid'))
    optimizer= hp.Choice('optimizer', values= ['adam', 'sgd', 'rmsprop', 'adadelta',])

    model.compile(optimizer = optimizer,
                  loss = 'binary_crossentropy', metrics= ['accuracy'])

    return model

In [133]:
tuner = kt.RandomSearch(build_model,
                        objective= 'val_accuracy',
                        max_trials= 5)

Reloading Tuner from .\untitled_project\tuner0.json


In [134]:
tuner.search(X_train, y_train, epochs = 5,validation_data= (X_test, y_test))

In [135]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [136]:
tuner.results_summary()

Results summary
Results in .\untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 0 summary
Hyperparameters:
optimizer: adam
Score: 0.7272727489471436

Trial 2 summary
Hyperparameters:
optimizer: rmsprop
Score: 0.7142857313156128

Trial 3 summary
Hyperparameters:
optimizer: sgd
Score: 0.6298701167106628

Trial 1 summary
Hyperparameters:
optimizer: adadelta
Score: 0.38311687111854553


In [137]:
model = tuner.get_best_models(num_models= 1)[0]

In [138]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [139]:
model.fit(X_train, y_train, batch_size= 32, epochs= 100, initial_epoch= 6, validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7215 - loss: 0.5858 - val_accuracy: 0.7662 - val_loss: 0.5724
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7443 - loss: 0.5599 - val_accuracy: 0.7662 - val_loss: 0.5508
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7524 - loss: 0.5397 - val_accuracy: 0.7727 - val_loss: 0.5336
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7638 - loss: 0.5226 - val_accuracy: 0.7857 - val_loss: 0.5204
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7638 - loss: 0.5091 - val_accuracy: 0.7987 - val_loss: 0.5112
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7638 - loss: 0.4976 - val_accuracy: 0.7987 - val_loss: 0.5018
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7671 - loss: 0.4889 - val_accuracy: 0.7857 - val_loss: 0.4946
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7687 - loss: 0.4815 - val_accuracy: 0.792